### Mercari PyCaret 분석기 클래스
1. TSV 데이터 로딩 지원
2. PyCaret setup, compare_models로 base model 탐색
3. 차원 축소(TF-IDF/Embedding 과 관련한 고차원 feature) 적용 가능
4. 단계별 진행 print 문구
5. tqdm 진행 표시
6. 모델 성능 지표 .json 저장
7. Submission CSV 저장
8. plot_model 시각화 저장 (../images/{model_name}_{timestamp}.png)

In [2]:
import pandas as pd
import numpy as np
import os
import json
import datetime
import gc
from tqdm import tqdm
import warnings

warnings.filterwarnings("ignore")

from pycaret.regression import *

from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.decomposition import TruncatedSVD

from kjh_mercari_analyzer import MercariPyCaretAnalyzer

In [3]:
# analyzer = MercariPyCaretAnalyzer()
# analyzer.load_data(train_file='train.tsv', test_file='test.tsv')
# analyzer.vectorize_text(method='tfidf', max_features=50000, n_components=100)
# analyzer.setup_pycaret()
# analyzer.find_base_model(sort_metric='R2')
# analyzer.save_metrics()
# analyzer.visualize_model(plots=['residuals','feature'])
# analyzer.predict_test(submission_file='submission.csv')

In [4]:
analyzer = MercariPyCaretAnalyzer(
    data_dir="../data", images_dir="../images", results_dir="../results"
)

In [5]:
analyzer.load_data()

📂 데이터 로딩 시작...
original data shape : train (1482535, 8), test (693359, 7)
✅ Price 외 결측치 처리 및 데이터 전처리 시작...
Price NaN count: 0
Final Price NaN count: 0
Train length: 1481661

Train head:
   train_id                                 name  item_condition_id  \
0         0  MLB Cincinnati Reds T Shirt Size XL                  3   
1         1     Razer BlackWidow Chroma Keyboard                  3   
2         2                       AVA-VIV Blouse                  1   
3         3                Leather Horse Statues                  1   
4         4                 24K GOLD plated rose                  1   

                                       category_name brand_name     price  \
0                                  Men/Tops/T-shirts    Unknown  2.397895   
1  Electronics/Computers & Tablets/Components & P...      Razer  3.970292   
2                        Women/Tops & Blouses/Blouse     Target  2.397895   
3                 Home/Home Décor/Home Décor Accents    Unknown  3.583519   
4 

In [6]:
analyzer.apply_undersampling(method="stratified", target_size=100000, n_bins=10)


🎯 Undersampling 시작...
   방법: stratified
   원본 크기: 1,481,661
   목표 크기: 100,000
   샘플링 비율: 6.75%
✅ Undersampling 완료
   최종 크기: 740,828
   실제 샘플링 비율: 50.00%
   제거된 샘플: 740,833

📊 샘플링 후 가격 분포:
   평균: 2.9809
   중앙값: 2.8904
   표준편차: 0.7462
   최소: 1.3863
   최대: 7.6034


In [7]:
analyzer.vectorize_text(method="tfidf", max_features=50000, n_components=100)

📝 텍스트 벡터화 및 차원 축소 시작...


Text columns:   0%|          | 0/2 [00:00<?, ?it/s]

▶ 컬럼: name
   ▪ 차원 축소 완료: 100 components


Text columns:  50%|█████     | 1/2 [02:44<02:44, 164.32s/it]

▶ 컬럼: item_description
   ▪ 차원 축소 완료: 100 components


Text columns: 100%|██████████| 2/2 [09:13<00:00, 276.88s/it]


✅ 벡터화 + 차원 축소 + 카테고리 인코딩 완료: train (740828, 205), test (693359, 205)


In [8]:
analyzer.setup_pycaret()

🔧 PyCaret setup 시작...


,Description,Value
0,Session id,23
1,Target,price
2,Target type,Regression
3,Original data shape,"(740828, 206)"
4,Transformed data shape,"(740828, 216)"
5,Transformed train set shape,"(518579, 216)"
6,Transformed test set shape,"(222249, 216)"
7,Numeric features,200
8,Categorical features,5
9,Preprocess,True


✅ PyCaret setup 완료


In [9]:
analyzer.find_base_model(sort_metric="R2")

🔍 Base model 탐색 시작...


,Model,MAE,MSE,RMSE,R2,RMSLE,MAPE,TT (Sec)
et,Extra Trees Regressor,0.4076,0.2944,0.5426,0.4711,0.1345,0.1450,1304.2680
xgboost,Extreme Gradient Boosting,0.4239,0.3095,0.5563,0.4441,0.1379,0.1502,12.3900
rf,Random Forest Regressor,0.4242,0.3148,0.5611,0.4346,0.1392,0.1513,2544.2910
catboost,CatBoost Regressor,0.3702,0.2624,0.4860,0.4287,0.1208,0.1315,100.2640
lightgbm,Light Gradient Boosting Machine,0.4424,0.3329,0.5770,0.4020,0.1431,0.1573,20.0330
gbr,Gradient Boosting Regressor,0.4816,0.3914,0.6256,0.2970,0.1546,0.1716,756.1810
lr,Linear Regression,0.4916,0.4084,0.6390,0.2665,0.1577,0.1746,78.9850
ridge,Ridge Regression,0.4916,0.4084,0.6390,0.2665,0.1577,0.1746,14.9810
lar,Least Angle Regression,0.4916,0.4084,0.6390,0.2665,0.1577,0.1746,62.8550
br,Bayesian Ridge,0.4916,0.4084,0.6390,0.2665,0.1577,0.1746,15.7000


🏆 Best model 선택 완료: ExtraTreesRegressor(n_jobs=-1, random_state=23)


ExtraTreesRegressor(n_jobs=-1, random_state=23)

In [10]:
analyzer.save_metrics()

KeyError: 'Label'

In [ ]:
analyzer.visualize_model(plots=["residuals", "feature"])

In [ ]:
analyzer.predict_test(submission_file="submission.csv")

In [ ]:
# analyzer.setup_pycaret()

🔧 PyCaret setup 시작...


TypeError: setup() got an unexpected keyword argument 'silent'